## Task 3: Multimodal ML – Housing Price Prediction Using Images + Tabular Data
#### Objective:
Predict housing prices using both structured data and house images.
#### Dataset:
Housing Sales Dataset + Custom Image Dataset (your own or any public source)
#### Instructions:
● Use CNNs to extract features from images

● Combine extracted image features with tabular data

● Train a model using both modalities

● Evaluate performance using MAE and RMSE

#### Skills Gained:
● Multimodal machine learning

● Convolutional Neural Networks (CNNs)

● Feature fusion (image + tabular)

● Regression modeling and evaluation

Step 1: Required Libraries

In [1]:
import pandas as pd
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout, concatenate, Input
from tensorflow.keras.optimizers import Adam

Step 2: Dummy Dataset Creation (For Testing)

In [2]:
# 1. Generate Dummy Tabular Data
data = {
    'area': np.random.randint(500, 5000, 100),
    'bedrooms': np.random.randint(1, 6, 100),
    'bathrooms': np.random.randint(1, 4, 100),
    'price': np.random.randint(50000, 500000, 100)
}
df = pd.DataFrame(data)

# 2. Generate Dummy Images
images = np.random.rand(100, 64, 64, 3) #100 images RGB

Step 3: Multimodal Model Design

In [3]:
#Branch01: MLP for Tabular Data
input_tab = Input(shape=(3,)) # area, bedrooms, bathrooms
x = Dense(16, activation="relu")(input_tab)
x = Dense(8, activation="relu")(x)
mlp_branch = Model(inputs=input_tab, outputs=x)

In [5]:
#Branch02: CNN for Images
input_img = Input(shape=(64, 64, 3))
y = Conv2D(16, (3, 3), activation="relu", padding="same")(input_img)
y = MaxPooling2D(pool_size=(2, 2))(y)
y = Flatten()(y)
y = Dense(8, activation="relu")(y)
cnn_branch = Model(inputs=input_img, outputs=y)

In [6]:
#FUSION: Combine Both Branches
combined = concatenate([mlp_branch.output, cnn_branch.output])

#Final Regression Layers
z = Dense(4, activation="relu")(combined)
z = Dense(1, activation="linear")(z) # Linear for Regression


In [7]:
# Final Model
model = Model(inputs=[mlp_branch.input, cnn_branch.input], outputs=z)
model.compile(optimizer=Adam(learning_rate=1e-3), loss='mse', metrics=['mae'])

Step 4: Training & Evaluation (MAE/RMSE)

In [8]:
# Training
X_tab = df[['area', 'bedrooms', 'bathrooms']].values
Y = df['price'].values
model.fit([X_tab, images], Y, epochs=10, batch_size=8, verbose=1)

Epoch 1/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 90124632064.0000 - mae: 271332.4062
Epoch 2/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 89990889472.0000 - mae: 271091.3125
Epoch 3/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 89801859072.0000 - mae: 270747.0000
Epoch 4/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 89503293440.0000 - mae: 270191.0312
Epoch 5/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 89023004672.0000 - mae: 269324.9375
Epoch 6/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 88310128640.0000 - mae: 268022.6875
Epoch 7/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 87303757824.0000 - mae: 266127.1562
Epoch 8/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 85884379136.0000 - mae: 263459.2500
Epoch 9/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 84016054272.0000 - mae: 259864.8438
Epoch 10/10
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 81551933440.0000 - mae: 255129.6250


In [9]:
# Model Evaluation/Evaluation Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error #Use Regression Metrics

predictions = model.predict([X_tab, images])
mae = mean_absolute_error(Y, predictions)
rmse = np.sqrt(mean_squared_error(Y, predictions))

print(f"Model Evaluation Results")
print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
Model Evaluation Results
MAE: 251938.73
RMSE: 282771.02
